In [ ]:
# https://arxiv.org/html/2411.10541v1/
# https://arxiv.org/pdf/2411.10541

In [ ]:
%pip install --upgrade langchain langchain-community langchain-core langchainhub langchain-qdrant langchain-ollama langchain-text-splitters langsmith

In [ ]:
json_file_path = './products_v4.json'

In [ ]:
import json
#read the content of json file
with open(json_file_path, 'r') as f:
    products_json = json.load(f)

In [ ]:
for product in products_json[:5]:
  print(product)

In [ ]:
def display(value):
  if value is None or len(value) == 0:
    return 'Not specified'
  return value

def display_list(lst):
  if lst is None or len(lst) == 0:
    return 'Not specified'
  return ', '.join(lst)

In [ ]:
def generate_product_profile(product):
  # Format the product profile
  product_profile = f"""
  # {product['Name']} - {product['Category']}
  ---

  ## Product Overview
  - Name: {display(product["Name"])}
  - Brand: {display(product["Brand"])}
  - Category: {display(product["Category"])}
  - Price: {display(product["Price"])} (in LKR)
  - Natural: {product["Natural"]}

  ## Ingredients
  - Key Ingredients: {display_list(product['Key Ingredients'])}
  - Concentrations: {display_list(product['Concentrations'])}
  - Full Ingredient List: {display_list(product['Ingredients'])}

  ## Benefits and Claims
  - Benefits: {display_list(product['Benefits'])}
  - Claims: {display_list(product['Claims'])}

  ## Usage and Application
  - Usage: {display(product['Usage'])}
  - Application Tips: {display(product['Application Tips'])}

  ## Skin Suitability
  - Suitable for Skin Types: {display_list(product['Skin Type'])}
  - Addresses Skin Concerns: {display_list(product['Skin Concerns'])}
  - For Sensitive Skin: {display(product['For Sensitive Skin'])}

  ## Safety Information
  - Potential Side Effects: {display_list(product['Potential Side Effects'])}
  - Allergens: {display_list(product['Allergens'])}

  ## Reviews and Ratings
  - Average Rating: {product['Average Rating']}/5
  - Customer Reviews:
  """
  # Append customer reviews with consistent indentation
  if product['Customer Reviews']:
    for i, review in enumerate(product['Customer Reviews']):
      if i == 0:
        product_profile += f"  - \"{review['review']}\" - {review['rating']} stars\n"
      else:
        product_profile += f"    - \"{review['review']}\" - {review['rating']} stars\n"
  else:
    product_profile += "    No customer reviews available.\n"

  # Append expert review
  product_profile += f"  - Expert Review: {display(product['Expert Review'])}"

  return product_profile

In [ ]:
product_profile = generate_product_profile(products_json[0])
print(product_profile)

In [ ]:
product_docs = []

for product in products_json:
  # Add formatted profile to the list
  product_docs.append({
    "template": generate_product_profile(product),
    "metadata": {
      "id": product["Id"],
      "name": product["Name"],
      "brand": product["Brand"],
      "category": product["Category"],
      "price": "LKR " + product["Price"]
    }
  })

In [ ]:
len(product_docs)

In [ ]:
product_docs[0]

In [ ]:
print(product_docs[0]["template"])

In [ ]:
from IPython.display import display, Markdown

display(Markdown(product_docs[0]["template"]))

### Generate engaging product descriptions using product profiles

In [ ]:
product_copywrite_prompt = """
You are an expert product copywriter. Based on the given product profile, write a professional and engaging product description that covers all the essential details, ingredients, benefits, usage instructions, and any special considerations.
Create a summary description of the skincare product based on the given structured product details.
Use only then given product information and phrase them as use see fit.
Don't remove any vital information from the product profile.
Be accurate and through.

Product Profile:
{product_profile}

Product Description:
"""

### Product Copywriter Agent

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_ollama import ChatOllama

def product_copywriter_agent(product_profile, prompt_template):
  prompt = PromptTemplate(input_variables=["product_profile"], template=prompt_template)

  agent = ChatOllama(
    base_url="http://216.81.248.11:11434",
    model="llama3.3:70b-instruct-q4_K_M",
    temperature=0,
  )

  product_copywrite_chain = LLMChain(llm=agent, prompt=prompt)

  product_description = product_copywrite_chain.run(product_profile=product_profile)

  return product_description

In [ ]:
product_docs_1 = product_docs[:25]
product_docs_2 = product_docs[25:50]
product_docs_3 = product_docs[50:75]
product_docs_4 = product_docs[75:]

In [ ]:
product_descriptions = []

for doc in product_docs_4:
  agent_response = product_copywriter_agent(doc["template"], product_copywrite_prompt)
  product_description = {
      "id": doc["metadata"]["id"],
      "metadata": doc["metadata"],
      "content": agent_response
  }
  product_descriptions.append(product_description)
  print(f"Product description for product {doc['metadata']['id']} generated")

In [ ]:
len(product_descriptions)

In [ ]:
output_file = "./product_descriptions.json"

In [ ]:
def append_to_json(file_path, new_data):
    # Read existing data
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError):
        data = []  # Initialize as list if file doesn't exist or is empty

    # Append new data
    data.append(new_data)

    # Write back to file
    with open(file_path, 'w') as file:
        json.dump(data, file, indent=4)

In [ ]:
append_to_json(output_file, product_descriptions)